# Module 8 Assignment: Give Your Persona Agent Memory With Sessions

Lab 7 added a `SQLiteSession` to Study Buddy: first in memory, to fix two `Runner.run` calls that
did not remember each other, then file backed, so the conversation survives a kernel restart. The
Gradio `chat()` function got simpler in the process, no more rebuilding `history` by hand.

This assignment makes you do the same thing to your **own** Persona Agent, the one you rebuilt on
the OpenAI Agents SDK in the Module 7 assignment. Nothing about your agent's persona, notes, or
three tools needs to change here: you are giving it memory, not redesigning it.

**What you'll build:**
1. Your Module 7 assignment's Agent, Azure configuration, and three tools, carried over unchanged
2. A quick demonstration that two separate `Runner.run` calls do not remember each other
3. A file backed `SQLiteSession` that fixes it
4. A simplified Gradio `chat()` function that uses `session=session` instead of rebuilding `history`

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 7 made the same kind of change to Study Buddy, and do
the analogous thing to your own agent.

## 0. Install the packages (run once)

Same as Lab 7: no new package beyond what your Module 7 assignment already installed.
`SQLiteSession` ships with `openai-agents` itself.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Using Python 3.14.7 environment at: C:\IT3025C\Lab7\Lab7InClass\.venv
Resolved 79 packages in 362ms
Checked 79 packages in 7ms


## 1. Imports

**TODO:** Copy the imports from your Module 7 assignment, plus one new name: `SQLiteSession`,
imported from `agents` alongside `Agent` and `Runner`, the same way Lab 7, Step 1 added it.

In [3]:
# TODO: copy your imports from the Module 7 assignment, plus SQLiteSession from agents
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
)
from pypdf import PdfReader
import gradio as gr

## 2. Load your Azure OpenAI credentials from `.env`

Identical to the Module 7 assignment. Sessions do not change which provider you talk to.

In [4]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from your Module 7 assignment, unchanged: `set_tracing_disabled(True)`,
your `AsyncAzureOpenAI` client, and `OpenAIChatCompletionsModel`. A session is a feature of
`Runner.run`, not of the model or the client, so nothing here needs to change.

In [5]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like your Module 7 assignment
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. Your persona, notes, and three tools

**TODO:** Bring over your persona, your notes, and all three of your `@function_tool` decorated
tools from the Module 7 assignment, unchanged. None of them need to change for this assignment: a
session only affects how conversation history is stored and replayed, not what a tool does.

In [11]:
# TODO: persona = "..." (a parenthesized multi-line string), from your Module 7 assignment
# TODO: if your persona uses notes, reader = PdfReader("notes/notes.pdf"); notes = ""; loop over
#       reader.pages and build up notes the same way Lab 6 did, if applicable
persona = (
    "Study Buddy, an upbeat and encouraging AI study coach. Study Buddy keeps quizzes short and "
    "fun, genuinely celebrates every right answer, and never makes a student feel bad about "
    "getting something wrong - just curious to help them learn it."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:300], "...")

Study Buddy, an upbeat and encouraging AI study coach. Study Buddy keeps quizzes short and fun, genuinely celebrates every right answer, and never makes a student feel bad about getting something wrong - just curious to help them learn it.
Bloodborne All Bosses Checklist
slaughter all of them
Regular Bosses
Cleric Beast
Father Gascoigne
Blood Starved Beast
Vicar Amelia
Witch of Hemwitch
Shadow of Yharman
Rom Vacuos Spider
Martyr Logarius
Darkbeast Paarl
The One Unborn
Celestial Emissary
Ebrietas
Amygdala
Micolash
Made with ChecklistMa ...


In [12]:
# TODO: your first tool, decorated with @function_tool, from your Module 7 assignment
LOG_PATH = "missed_questions.txt"

@function_tool
def log_missed_question_tool(question: str) -> str:
    """Record a study question the student got wrong or was unsure about, so they can review it later.

    Args:
        question: The exact question the student missed or was unsure about.
    """
    print(f"Tool called to log a missed question: {question}")
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

print("Tool ready:", log_missed_question_tool.name)

Tool ready: log_missed_question_tool


In [13]:
# TODO: your second tool, decorated with @function_tool, from your Module 7 assignment
OPENTDB_CATEGORIES = {"general_knowledge": 9, "science_computers": 18}

@function_tool
def fetch_bonus_question_tool(category: Optional[Literal["general_knowledge", "science_computers"]] = None) -> str:
    """Fetch a fun bonus trivia question from an external trivia database, for when the student asks for a bonus question or a break from the study notes.

    Args:
        category: Which trivia category to pull a bonus question from, if the student asked for one.
    """
    print(f"Tool called to fetch a bonus trivia question (category={category})")
    params = {"amount": 1, "type": "multiple"}
    if category in OPENTDB_CATEGORIES:
        params["category"] = OPENTDB_CATEGORIES[category]

    try:
        response = requests.get("https://opentdb.com/api.php", params=params, timeout=20)
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"fetch_bonus_question_tool network error: {e}")
        return "The trivia API is temporarily unavailable - just skip the bonus question and continue the quiz."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now - just skip it and continue the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])
    return f"Bonus question: {question} (correct answer: {answer})"

print("Tool ready:", fetch_bonus_question_tool.name)

Tool ready: fetch_bonus_question_tool


In [14]:
# TODO: your third tool, decorated with @function_tool, from your Module 7 assignment
@function_tool
def check_progress_tool() -> str:
    """Check how many study questions the student has missed so far, for when they ask about their progress."""
    print("Tool called to check quiz progress")
    try:
        with open(LOG_PATH, encoding="utf-8") as f:
            missed = [line for line in f if line.strip()]
    except FileNotFoundError:
        missed = []

    if not missed:
        return "No questions have been missed yet - a clean sheet so far!"
    return f"So far, {len(missed)} question(s) have been logged as missed."

print("Tool ready:", check_progress_tool.name)

Tool ready: check_progress_tool


## 5. Build your Agent

**TODO:** Copy this from your Module 7 assignment, unchanged: `name`, `instructions`, your `tools`
list, and `model=azure_model`.

In [15]:
# TODO: instructions = f"""..."""  (your # Your role / # Context / # Rules structured template,
#       using {persona} and {notes}, from your Module 7 assignment)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)
instructions = f"""
# Your role

You are {persona}

You quiz a student on the study notes below.

# Context

Here are the study notes you are quizzing the student on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

If the student's answer was wrong, or they say "I don't know", you must record it as a missed
question using log_missed_question_tool before moving on to the next question.

If the student asks for a bonus question, or seems to want a break from the notes, use
fetch_bonus_question_tool to get one - this question does not need to relate to the notes.

If the student asks how they are doing, or asks for a progress check, use check_progress_tool to
tell them how many questions they've missed so far.

IMPORTANT: only ask quiz questions from the notes above. If the student asks about something
outside the notes (and it isn't a bonus question or a progress check), say that it isn't covered
in this study set.
"""

study_buddy = Agent(
    name="Study Buddy",
    instructions=instructions,
    tools=[log_missed_question_tool, fetch_bonus_question_tool, check_progress_tool],
    model=azure_model,
)

print("Agent built:", study_buddy.name)

Agent built: Study Buddy


## 6. Confirm the problem: two calls that do not know each other

**TODO:** Run your agent twice in a row with `await Runner.run(your_agent, "...")`, the same way
Lab 7, Step 7 did with Study Buddy. Pick a first message that gives the agent some information (for
example, ask it a question whose answer it needs to remember), then a second message that only
makes sense if the agent remembers the first (for example, "what did I just ask you?" or "say that
again"). Confirm the second reply shows the agent has no idea what you mean.

In [16]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)
result = await Runner.run(study_buddy, "Hi! Can you quiz me with one question from my notes?")
print(result.final_output)

You got it! 🎮 First question: which boss is listed first under **Regular Bosses**?


In [17]:
# TODO: result = await Runner.run(your_agent, "...a follow up that only makes sense with memory...")
# TODO: print(result.final_output)
result = await Runner.run(study_buddy, "Actually, I don't know the answer to that one. Can you read it back so i can try again?")
print(result.final_output)

No problem at all — let’s give it another shot. I can repeat the question, but I need to stick to one quiz question at a time from the study set.

Here it is again:

Which boss is listed immediately after Micolash in the Regular Bosses section?


## 7. Fix it with a file backed `SQLiteSession`

**TODO:** Create a `SQLiteSession`, the same way Lab 7, Step 9 did: give it a `session_id` string,
and a `db_path` so the conversation is written to a file and survives a kernel restart. Then repeat
the same two messages from Step 6, this time passing `session=your_session` to both calls, and
confirm the second reply now correctly remembers the first.

In [26]:
# TODO: your_session = SQLiteSession("your_session_id", "your_conversation_history.db")
study_session = SQLiteSession("demo_conversation")

result = await Runner.run(
    study_buddy,
    "Hi! Can you quiz me with one question from my notes?",
    session=study_session,
)
print(result.final_output)

Absolutely! 🎮 First question: Which boss comes right after Cleric Beast in the Regular Bosses list?


In [27]:
# TODO: result = await Runner.run(your_agent, "...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(
    study_buddy,
    "Actually, I don't know the answer to that one.",
    session=study_session,
)
print(result.final_output)

Tool called to log a missed question: Which boss comes right after Cleric Beast in the Regular Bosses list?
No worries at all — that one was Father Gascoigne. 🌟 He comes right after Cleric Beast in the Regular Bosses list.

Next question: Which boss is listed right after Vicar Amelia?


In [28]:
# TODO: result = await Runner.run(your_agent, "...same follow up as Step 6...", session=your_session)
# TODO: print(result.final_output)
items = await study_session.get_items()
print(f"{len(items)} items stored in this session")
for item in items:
    print(item.get("role"), "-", str(item.get("content"))[:80])

6 items stored in this session
user - Hi! Can you quiz me with one question from my notes?
assistant - [{'annotations': [], 'text': 'Absolutely! 🎮 First question: Which boss comes rig
user - Actually, I don't know the answer to that one.
None - None
None - None
assistant - [{'annotations': [], 'text': 'No worries at all — that one was Father Gascoigne.


## 8. Give it a real chat UI with Gradio

**TODO:** Copy this skeleton from Lab 7, Step 11 exactly, structure and all. Notice what is
different from your Module 7 assignment's version: `history` is no longer rebuilt inside `chat()`
at all, `session=your_session` already gives `Runner.run` everything it needs to remember the
conversation.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [30]:
async def chat(message, history):
    # TODO: result = await Runner.run(your_agent, message, session=your_session)
    # TODO: return result.final_output
    pass
    result = await Runner.run(study_buddy, message, session=study_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called to log a missed question: Which DLC boss is listed right after Ludwig, the Holy Blade?


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. In Step 6, what exactly did your agent get wrong on the second call, and why? Point to the
   specific piece of information it had no way of knowing.
2. Compare your Gradio `chat()` function here to the one in your Module 7 assignment. What code did
   you get to delete, and what does `session=your_session` do in its place?
3. Your `your_session` was created with a `db_path`. If you had left `db_path` out entirely, what
   would still work the same, and what would break the next time you restarted the kernel?
4. `SQLiteSession` also has a `pop_item()` method, demonstrated in Lab 7, Step 10, for undoing the
   last turn. Describe one realistic situation in your own agent's domain where you would want to
   call it.

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Remove any `.db` file your session created; it may contain real conversation content.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm Step 6 shows the agent failing to remember, and Step 7 shows the same scenario working
  correctly once `session=your_session` is added.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module8Assignment_JaneDoe.ipynb`, then submit
  it.

1. In Step 6, what exactly did your agent get wrong on the second call, and why? Point to the specific piece of information it had no way of knowing.

In the second call, my agent got the original asked question wrong. It logged a question that had nthing to do with the one that I had said I didn't know. It had no way of knowing what the previous question was since it hadn't been set up with a memory.

2. Compare your Gradio `chat()` function here to the one in your Module 7 assignment. What code did you get to delete, and what does `session=your_session` do in its place?

what is different is we removed the role, user, and content persona identifiers. Instead of manually defining parameters of the study buddy in the gradio chat, we can ask it to call from a preset definition. Session=your_session takes its place to save against having to reset those conditions.

3. Your `your_session` was created with a `db_path`. If you had left `db_path` out entirely, what would still work the same, and what would break the next time you restarted the kernel?

The persistence of the memory. The code and agent would still run the same during actual use. Without the db_path, the memory would crash and be lost when the kernel dies. Anything logged in a previous session is lost and the chat must be started from scratch.

4. `SQLiteSession` also has a `pop_item()` method, demonstrated in Lab 7, Step 10, for undoing the last turn. Describe one realistic situation in your own agent's domain where you would want to call it.

With this articular study guide, a user culd give a correct answer but have mistyped it, resulting in a negative score when the answer was echnicaly correct. In this instance the student would want to undo the negative mark and fix the error.
